# De KNIME a Python: pronóstico como nodos

En KNIME pensamos en un flujo: datos de entrada, nodos configurados y salidas revisables.

En este notebook vamos a escribir esa misma idea en Python:

- **Clase** = definición de un nodo KNIME.
- **Objeto** = nodo específico arrastrado al lienzo.
- **Encapsulamiento** = lo que vive dentro del nodo y su ventana de configuración.

## 1. Datos de entrada

Estos datos representan la tabla que llega al nodo de pronóstico.

Guardrail de clase: el nodo solo puede usar información disponible **antes** de pronosticar.

In [1]:
datos_clima = [
    {"dia": "Lunes", "temperatura_c": 25},
    {"dia": "Martes", "temperatura_c": 34},
    {"dia": "Domingo", "temperatura_c": 18},
]

print(datos_clima)

[{'dia': 'Lunes', 'temperatura_c': 25}, {'dia': 'Martes', 'temperatura_c': 34}, {'dia': 'Domingo', 'temperatura_c': 18}]


## 2. El flujo sin nodos: código procedimental desastroso

Este script funciona, pero mezcla datos, parámetros, reglas y reporte.

En KNIME sería como tener un flujo enorme con nodos duplicados y configuraciones escondidas.

In [2]:
print("--- SCRIPT PROCEDURAL ---")

for fila in datos_clima:
    temperatura = fila["temperatura_c"]
    dia = fila["dia"]
    carga_base_norte = 500

    if dia == "Domingo" or dia == "Sabado":
        demanda = carga_base_norte * 0.75
    else:
        demanda = carga_base_norte

    if temperatura > 30:
        demanda = demanda + (temperatura - 30) * 12.5

    print(f"Subestacion Norte | {dia} | {temperatura} C | {demanda} MW")

for fila in datos_clima:
    temperatura = fila["temperatura_c"]
    dia = fila["dia"]
    carga_base_sur = 850

    if dia == "Domingo" or dia == "Sabado":
        demanda = carga_base_sur * 0.75
    else:
        demanda = carga_base_sur

    if temperatura > 32:
        demanda = demanda + (temperatura - 32) * 15.1

    print(f"Subestacion Sur | {dia} | {temperatura} C | {demanda} MW")

--- SCRIPT PROCEDURAL ---
Subestacion Norte | Lunes | 25 C | 500 MW
Subestacion Norte | Martes | 34 C | 550.0 MW
Subestacion Norte | Domingo | 18 C | 375.0 MW
Subestacion Sur | Lunes | 25 C | 850 MW
Subestacion Sur | Martes | 34 C | 880.2 MW
Subestacion Sur | Domingo | 18 C | 637.5 MW


### ¿Qué problema vemos?

- La configuración está escondida dentro del ciclo.
- La misma regla se copia varias veces.
- No hay una revisión clara de entradas.
- El resultado no deja suficiente rastro para auditoría.

## 3. Definir un nodo en Python

La clase `NodoDemandaSubestacion` es la definición del nodo.

Su ventana de configuración vive en `__init__`: nombre, carga base, umbral de calor y factores.

In [3]:
class NodoDemandaSubestacion:
    def __init__(
        self,
        nombre,
        carga_base_mw,
        umbral_calor_c,
        factor_calor_mw,
        factor_fin_semana,
    ):
        self.nombre = nombre
        self.carga_base_mw = carga_base_mw
        self.umbral_calor_c = umbral_calor_c
        self.factor_calor_mw = factor_calor_mw
        self.factor_fin_semana = factor_fin_semana

    def revisar_configuracion(self):
      if self.carga_base_mw <= 0:
        print(f"ALERTA {self.nombre}: carga base no valida")

      if self.factor_fin_semana <= 0 or self.factor_fin_semana > 1:
        print(f"ALERTA {self.nombre}: factor de fin de semana no valido")

    def procesar_fila(self, fila):
        dia = fila["dia"]
        temperatura_c = fila["temperatura_c"]
        demanda_mw = self.carga_base_mw
        regla = "base"

        if dia in ["Sabado", "Domingo"]:
            demanda_mw = demanda_mw * self.factor_fin_semana
            regla = regla + " + fin_semana"

        if temperatura_c > self.umbral_calor_c:
            exceso_calor = temperatura_c - self.umbral_calor_c
            demanda_mw = demanda_mw + exceso_calor * self.factor_calor_mw
            regla = regla + " + ajuste_calor"

        resultado = {
            "subestacion": self.nombre,
            "dia": dia,
            "temperatura_c": temperatura_c,
            "demanda_mw": round(demanda_mw, 2),
            "regla": regla,
        }

        return resultado

    def procesar_tabla(self, tabla):
        resultados = []

        for fila in tabla:
            resultado = self.procesar_fila(fila)
            resultados.append(resultado)

        return resultados

## 4. Crear objetos: nodos específicos en el lienzo

Cada objeto es un nodo ya configurado.

Todos usan la misma definición, pero cada subestación tiene sus propios parámetros.

In [4]:
nodo_norte = NodoDemandaSubestacion(
    nombre="Norte",
    carga_base_mw=500,
    umbral_calor_c=30,
    factor_calor_mw=12.5,
    factor_fin_semana=0.75,
)

nodo_sur = NodoDemandaSubestacion(
    nombre="Sur",
    carga_base_mw=850,
    umbral_calor_c=32,
    factor_calor_mw=15.1,
    factor_fin_semana=0.75,
)

nodo_este = NodoDemandaSubestacion(
    nombre="Este",
    carga_base_mw=420,
    umbral_calor_c=28,
    factor_calor_mw=10.0,
    factor_fin_semana=0.80,
)

parque_subestaciones = [nodo_norte, nodo_sur, nodo_este]

for nodo in parque_subestaciones:
    nodo.revisar_configuracion()

print("Nodos listos:", [nodo.nombre for nodo in parque_subestaciones])

Nodos listos: ['Norte', 'Sur', 'Este']


## 5. Pilar: Reproducibilidad

La configuración está visible. Si otro equipo usa los mismos datos y los mismos nodos, debe obtener el mismo resultado.

In [5]:
print("--- CONFIGURACION VISIBLE ---")

for nodo in parque_subestaciones:
    print(
        nodo.nombre,
        nodo.carga_base_mw,
        nodo.umbral_calor_c,
        nodo.factor_calor_mw,
        nodo.factor_fin_semana,
    )

--- CONFIGURACION VISIBLE ---
Norte 500 30 12.5 0.75
Sur 850 32 15.1 0.75
Este 420 28 10.0 0.8


## 6. Pilar: Evitar Data Leakage

Antes de procesar, revisamos que la tabla solo tenga columnas disponibles antes del pronóstico.

En esta práctica permitimos `dia` y `temperatura_c`.

In [6]:
columnas_permitidas = ["dia", "temperatura_c"]

for fila in datos_clima:
    for columna in fila:
        if columna not in columnas_permitidas:
            print("ALERTA: columna no permitida antes del pronostico:", columna)

print("Revision de leakage terminada")

Revision de leakage terminada


## 7. Pilar: Sanity Checks

Antes de confiar en el resultado, revisamos que las entradas tengan sentido físico.

In [7]:
dias_validos = ["Lunes", "Martes", "Miercoles", "Jueves", "Viernes", "Sabado", "Domingo"]

for fila in datos_clima:
    dia = fila["dia"]
    temperatura_c = fila["temperatura_c"]

    if dia not in dias_validos:
        print("ALERTA: dia no valido", dia)

    if temperatura_c < -20 or temperatura_c > 55:
        print("ALERTA: temperatura fuera de rango", temperatura_c)

print("Sanity checks terminados")

Sanity checks terminados


## 8. Procesar la tabla completa

El flujo principal ya no toca la fórmula.

Solo entrega la tabla a cada nodo y recibe resultados.

In [8]:
reporte = []

for nodo in parque_subestaciones:
    resultados_del_nodo = nodo.procesar_tabla(datos_clima)

    for resultado in resultados_del_nodo:
        reporte.append(resultado)

print("--- REPORTE TRAZABLE ---")

for fila in reporte:
    print(fila)

--- REPORTE TRAZABLE ---
{'subestacion': 'Norte', 'dia': 'Lunes', 'temperatura_c': 25, 'demanda_mw': 500, 'regla': 'base'}
{'subestacion': 'Norte', 'dia': 'Martes', 'temperatura_c': 34, 'demanda_mw': 550.0, 'regla': 'base + ajuste_calor'}
{'subestacion': 'Norte', 'dia': 'Domingo', 'temperatura_c': 18, 'demanda_mw': 375.0, 'regla': 'base + fin_semana'}
{'subestacion': 'Sur', 'dia': 'Lunes', 'temperatura_c': 25, 'demanda_mw': 850, 'regla': 'base'}
{'subestacion': 'Sur', 'dia': 'Martes', 'temperatura_c': 34, 'demanda_mw': 880.2, 'regla': 'base + ajuste_calor'}
{'subestacion': 'Sur', 'dia': 'Domingo', 'temperatura_c': 18, 'demanda_mw': 637.5, 'regla': 'base + fin_semana'}
{'subestacion': 'Este', 'dia': 'Lunes', 'temperatura_c': 25, 'demanda_mw': 420, 'regla': 'base'}
{'subestacion': 'Este', 'dia': 'Martes', 'temperatura_c': 34, 'demanda_mw': 480.0, 'regla': 'base + ajuste_calor'}
{'subestacion': 'Este', 'dia': 'Domingo', 'temperatura_c': 18, 'demanda_mw': 336.0, 'regla': 'base + fin_semana

## 9. Pilar: Trazabilidad

Cada fila del reporte responde:

- qué subestación la produjo;
- qué día y temperatura usó;
- qué demanda calculó;
- qué regla se activó.

In [9]:
print("--- LECTURA HUMANA DEL REPORTE ---")

for fila in reporte:
    mensaje = (
        f"{fila['subestacion']} | "
        f"{fila['dia']} | "
        f"{fila['temperatura_c']} C | "
        f"{fila['demanda_mw']} MW | "
        f"{fila['regla']}"
    )
    print(mensaje)

--- LECTURA HUMANA DEL REPORTE ---
Norte | Lunes | 25 C | 500 MW | base
Norte | Martes | 34 C | 550.0 MW | base + ajuste_calor
Norte | Domingo | 18 C | 375.0 MW | base + fin_semana
Sur | Lunes | 25 C | 850 MW | base
Sur | Martes | 34 C | 880.2 MW | base + ajuste_calor
Sur | Domingo | 18 C | 637.5 MW | base + fin_semana
Este | Lunes | 25 C | 420 MW | base
Este | Martes | 34 C | 480.0 MW | base + ajuste_calor
Este | Domingo | 18 C | 336.0 MW | base + fin_semana


## 10. Ejercicios de Refuerzo Progresivo

En esta sección aplicaremos el enfoque de 'Nodos' para resolver retos reales. Recuerda: no estamos programando scripts sueltos, estamos configurando herramientas reutilizables.

### Ejercicio 1: Configuración de la Subestación 'Oeste' (Nivel Inicial)

Imagina que el departamento de planeación te entrega la ficha técnica de una nueva subestación. Tu tarea es 'configurar el nodo' con estos parámetros específicos:

*   **Nombre:** "Oeste"
*   **Carga Base:** 610 MW
*   **Umbral de Calor:** 31°C (esta subestación es más moderna y resiste más calor).
*   **Factor Calor:** 13.0 MW por cada grado extra.
*   **Factor Fin de Semana:** 0.78 (representa una reducción del 22% en sábados y domingos).

**Instrucciones:**
1. Crea el objeto `nodo_oeste` configurando cada parámetro.
2. Ejecuta el método `.procesar_tabla(datos_clima)`.
3. Imprime los resultados y observa cómo el Martes (34°C) se activa el ajuste de calor.

In [10]:
# PASO 1: Configurar el nodo
nodo_oeste = NodoDemandaSubestacion(
    nombre="Oeste",
    carga_base_mw=610,
    umbral_calor_c=31,
    factor_calor_mw=13.0,
    factor_fin_semana=0.78
)

# PASO 2: Procesar la información
resultado_oeste = nodo_oeste.procesar_tabla(datos_clima)

# PASO 3: Ver el reporte final
resultado_oeste
# print(f"--- REPORTE SUBESTACIÓN {nodo_oeste.nombre} ---")
# for fila in resultado_oeste:
#    print(fila)

[{'subestacion': 'Oeste',
  'dia': 'Lunes',
  'temperatura_c': 25,
  'demanda_mw': 610,
  'regla': 'base'},
 {'subestacion': 'Oeste',
  'dia': 'Martes',
  'temperatura_c': 34,
  'demanda_mw': 649.0,
  'regla': 'base + ajuste_calor'},
 {'subestacion': 'Oeste',
  'dia': 'Domingo',
  'temperatura_c': 18,
  'demanda_mw': 475.8,
  'regla': 'base + fin_semana'}]

### Ejercicio 2: El 'Semáforo' de Configuración (Nivel Medio)

En KNIME, si un nodo tiene un parámetro imposible, se pone en rojo. En Python, nuestra clase tiene el método `.revisar_configuracion()` para esto.

**Reto:**
1. Crea un nuevo nodo llamado `Sub_Prueba`.
2. Configúralo con un valor erróneo: por ejemplo, una `carga_base_mw` de `-100` o un `factor_fin_semana` de `1.5`.
3. Ejecuta el método `revisar_configuracion()` de ese objeto.
4. **Pregunta:** ¿Qué mensaje te dio el sistema? ¿Por qué es importante validar los datos antes de hacer el cálculo?

In [14]:
# PASO 1: Configurar el nodo
nodo_subprueba = NodoDemandaSubestacion(
    nombre="sub_prueba",
    carga_base_mw=-100,
    umbral_calor_c=31,
    factor_calor_mw=13.0,
    factor_fin_semana=1.001
)
nodo_subprueba.revisar_configuracion()
# PASO 2: Procesar la información
nodo_subprueba = nodo_subprueba.procesar_tabla(datos_clima)

# PASO 3: Ver el reporte final
nodo_subprueba
# print(f"--- REPORTE SUBESTACIÓN {nodo_oeste.nombre} ---")
# for fila in nodo_subprueba:
#    print(fila)

ALERTA sub_prueba: carga base no valida
ALERTA sub_prueba: factor de fin de semana no valido


[{'subestacion': 'sub_prueba',
  'dia': 'Lunes',
  'temperatura_c': 25,
  'demanda_mw': -100,
  'regla': 'base'},
 {'subestacion': 'sub_prueba',
  'dia': 'Martes',
  'temperatura_c': 34,
  'demanda_mw': -61.0,
  'regla': 'base + ajuste_calor'},
 {'subestacion': 'sub_prueba',
  'dia': 'Domingo',
  'temperatura_c': 18,
  'demanda_mw': -100.1,
  'regla': 'base + fin_semana'}]

### Ejercicio 3: Extendiendo el Nodo para el Frío (Nivel Experto)

Las subestaciones también consumen más energía cuando hace **demasiado frío** (menos de 5°C) por el uso de calefactores industriales.

**Reto:**
1. Crea una copia de la clase `NodoDemandaSubestacion` en una celda nueva.
2. Modifica el método `procesar_fila` para que si la temperatura es menor a 5°C, sume 10 MW a la demanda final.
3. Prueba tu nuevo código con un dato de temperatura de `2°C`.

*Tip: Usa una estructura parecida a la del calor:* `if temperatura_c < 5: demanda_mw = demanda_mw + 10`.

In [16]:
class NodoDemandaSubestacionExperto(NodoDemandaSubestacion):

    def procesar_fila(self, fila):
        dia = fila["dia"]
        temperatura_c = fila["temperatura_c"]
        demanda_mw = self.carga_base_mw
        regla = "base"

        # Regla de fin de semana
        if dia in ["Sabado", "Domingo"]:
            demanda_mw = demanda_mw * self.factor_fin_semana
            regla += " + fin_semana"

        # Regla de calor (Existente)
        if temperatura_c > self.umbral_calor_c:
            exceso_calor = temperatura_c - self.umbral_calor_c
            demanda_mw += exceso_calor * self.factor_calor_mw
            regla += " + ajuste_calor"

        # NUEVA REGLA: Ajuste por frío (Ejercicio 3)
        if temperatura_c < 5:
            demanda_mw += 10
            regla += " + ajuste_frio"

        return {
            "subestacion": self.nombre,
            "dia": dia,
            "temperatura_c": temperatura_c,
            "demanda_mw": round(demanda_mw, 2),
            "regla": regla
        }

# --- PRUEBA DEL EJERCICIO 3 ---
# 1. Configuramos el nodo experto
nodo_invierno = NodoDemandaSubestacionExperto(
    nombre="Norte_Invierno",
    carga_base_mw=500,
    umbral_calor_c=30,
    factor_calor_mw=12.5,
    factor_fin_semana=0.99
)

# 2. Creamos un dato de prueba con 2°C
dato_frio = [{"dia": "Lunes", "temperatura_c": 2}]

# 3. Procesamos y mostramos el resultado
resultado_frio = nodo_invierno.procesar_fila(dato_frio[0])
print("Resultado con 2°C:")
display(resultado_frio)

Resultado con 2°C:


{'subestacion': 'Norte_Invierno',
 'dia': 'Lunes',
 'temperatura_c': 2,
 'demanda_mw': 510,
 'regla': 'base + ajuste_frio'}